In [30]:
from __future__ import annotations


In [31]:
# input graph: this is the example from the slides deck used in class.
_ = float("inf")
g = [
    [_, _, _, 5, 1, _],
    [_, _, 20, 5, _, 10],
    [_, 20, _, 10, _, _],
    [5, 5, 10, _, _, 15],
    [1, _, _, _, _, 20],
    [_, 10, _, 15, 20, _],
]

In [32]:
def reach(s: int, graph: list[list[float]]) -> list[int]:
    """Finds every vertex reachable from s by following edges in graph.

    A worklist-based traversal: start with only s to process, and keep
    pulling one vertex at a time off the worklist, adding it to the
    reached list the first time it's seen, and queuing up its neighbors
    to process next. The worklist's pop() takes from the end (last in,
    first out), which affects the order vertices come out in, but not
    which vertices end up reachable.

    Parameters:
        s: the starting vertex.
        graph: an adjacency matrix; graph[u][v] is the edge weight
            between u and v, or the sentinel value graph[0][0]
            (typically infinity) if no edge exists.
    Returns:
        list[int]: every vertex reachable from s, including s itself,
        in the order they were first reached.
    """
    no_edge = graph[0][0]  # the sentinel value this matrix uses for "no edge"
    n = len(graph)
    reached = list()  # vertices confirmed reachable, in the order found
    bag = [s]  # the worklist: vertices still waiting to be processed
    while bag:
        u = bag.pop()
        if u not in reached:
            reached.append(u)
            # queue every neighbor u has an edge to, so the loop above
            # eventually processes them too
            for v in range(n):
                if graph[u][v] != no_edge:
                    bag.append(v)
    return reached

In [33]:
def count_label(graph: list[list[float]]) -> tuple[int, list[int]]:
    """Counts the connected components of graph and labels every vertex
    with the id of the component it belongs to.

    Builds on reach(s, graph): for each vertex not yet reached by an
    earlier call, start a new component, find everything reachable from
    it, and label all of those vertices with the new component's count.

    Parameters:
        graph: an adjacency matrix (same no-edge convention as reach).
    Returns:
        tuple[int, list[int]]: (count, comp) -- the number of
        components, and a list where comp[v] is the component label of
        vertex v.
    """
    # TODO: mark every vertex as unlabeled first. For each vertex not
    # yet marked, increment the component count, call reach() to find
    # everything connected to it, label all of those vertices with the
    # current count, and mark them. Return (count, comp).
    n = len(graph)
    # Initialize the comp labeling array, where comp[u] is the 
    # component label for vertex u
    comp = [ -1 ] * n
    # Counter for components
    counter = 0
    # List of vertices visited, so we can tell when we are in a 
    # new component.
    seen_before = []
    # Consider every vertex in the input graph
    for s in range(n):
        # Have we seen s before?
        if s not in seen_before:
            # Nope; it means we have just stumpled into a 
            # new component
            counter += 1
            # Find all the vertices reachable from s
            vertices_in_this_component = reach(s,graph)
            # Label these vertices with their component number
            for v in vertices_in_this_component:
                comp[v] = counter
            # Mark these vertices as seen before
            seen_before.extend(vertices_in_this_component)
    return (counter, comp) 

In [34]:
def initialize_mst(graph: list[list[float]]) -> list[list[float]]:
    """Builds an edgeless working copy of graph: same size as graph,
    every entry set to infinity (no edge).

    Parameters:
        graph: the input adjacency matrix.
    Returns:
        list[list[float]]: a new n x n matrix, n = len(graph), with
        every entry set to infinity -- this is T before any edges are
        added.
    """
    n: int = len(graph)
    no_edge: float = graph[0][0]
    return [[no_edge for _ in range(n)] for _ in range(n)]

In [35]:
def boruvka_mst(graph) -> list[list[float]]:
    """Builds a minimum spanning tree from graph using the safe-edge
    algorithm: start from an edgeless working copy, and repeatedly find
    and add a safe edge between two different components until only one
    component remains.

    Parameters:
        graph: the input weighted adjacency matrix.
    Returns:
        list[list]: a working-copy-shaped matrix containing only the
        edges of the minimum spanning tree.
    """
    no_edge = graph[0][0]  # the sentinel value this matrix uses for "no edge"
    T = initialize_mst(graph)
    n = len(graph)
    count, comp = count_label(T)
    while count > 1:
        # Initialize a list of safe edges for the components of this
        # iteration. Each component's safe edge is the minimum-weight
        # edge that connects it to a different component. If a component
        # has no edges to other components, its entry in this list will
        # remain None. We initialize to count+1 so that 1-based indexing 
        # (from 1 to count) can be used    
        safe_edge = [None] * (count+1)  
        # Consider every possible edge in the input graph:
        for u in range(n):
            for v in range(n):
                # Is there a [u,v] edge in the input graph? 
                if graph[u][v] != no_edge:
                    # Are u and v in diferent components of the output graph T? 
                    # If so, this edge is a candidate for being a safe edge.
                    if comp[u] != comp[v]:
                        # Good, [u,v] is a candidate safe edge. Let's see
                        # one component at a time, if it's an actual 
                        # safe edge for that component.
                        if safe_edge[comp[u]] is None:
                            # comp[u] current has no safe edge assigned to it.
                            # Make [u,v] the safe edge for comp[u]
                            safe_edge[comp[u]] = [u,v]
                        else:
                            # There is a safe edge already assigned to the component.
                            # Let's find its vertices.
                            u1, v1 = safe_edge[comp[u]]
                            # Now look up the weight of this edge in the
                            # input graph and compare it to the weight of th
                            # candidate edge [u,v]. If the candidate is
                            # lighter, make it the new safe edge for comp[u].
                            if graph[u][v] < graph[u1][v1]:
                                safe_edge[comp[u]] = [u,v]
                        # Let's do the same for the other component, comp[v]
                        if safe_edge[comp[v]] is None:
                            safe_edge[comp[v]] = [u,v]
                        else:
                            u1, v1 = safe_edge[comp[v]]
                            if graph[u][v] < graph[u1][v1]:
                                safe_edge[comp[v]] = [u,v]
        # Now we have a safe edge for each component. Add them to T.
        for edge in safe_edge:
            if edge is not None:
                u, v = edge
                T[u][v] = graph[u][v]
                T[v][u] = graph[v][u]
        # Recompute the number of components and their labels in T.
        count, comp = count_label(T)
    return T

In [36]:
t = boruvka_mst(g)
print(t)
no_edge = t[0][0]
n = len(t)
edges = [(u, v, t[u][v]) for u in range(n) for v in range(u + 1, n) if t[u][v] != no_edge]
total_weight = sum(weight for _, _, weight in edges)

print(f"Minimum spanning tree: {len(edges)} edges, total weight {total_weight}")
for u, v, weight in edges:
    print(f"  {u}-{v}: {weight}")

expected_weight = 31
match = "matches" if total_weight == expected_weight else "does NOT match"
print(f"\nExpected total from the hand-trace: {expected_weight} -- {match}")
print(t)

[[inf, inf, inf, 5, 1, inf], [inf, inf, inf, 5, inf, 10], [inf, inf, inf, 10, inf, inf], [5, 5, 10, inf, inf, inf], [1, inf, inf, inf, inf, inf], [inf, 10, inf, inf, inf, inf]]
Minimum spanning tree: 5 edges, total weight 31
  0-3: 5
  0-4: 1
  1-3: 5
  1-5: 10
  2-3: 10

Expected total from the hand-trace: 31 -- matches
[[inf, inf, inf, 5, 1, inf], [inf, inf, inf, 5, inf, 10], [inf, inf, inf, 10, inf, inf], [5, 5, 10, inf, inf, inf], [1, inf, inf, inf, inf, inf], [inf, 10, inf, inf, inf, inf]]
